In [89]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from tqdm import tqdm

from scipy.signal import convolve
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import r2_score, mean_squared_error

import utils
import paths as p

In [90]:
# glm_dir = Path(p.DATA_DIR) / 'glm_encoding'
glm_dir = Path('/Volumes/T7 Shield') / 'glm_encoding_colvolution'

model_dir = glm_dir / 'glm_models'
model_dir.mkdir(parents=True, exist_ok=True)

plots_dir = glm_dir / 'glm_plots'
plots_dir.mkdir(parents=True, exist_ok=True)

units_vetted = pd.read_csv(os.path.join(p.LOGS_DIR, 'units_vetted.csv'), index_col=0).sort_values('unit_id')

In [91]:
events, trials, spikes = utils.get_data_for_debugging(units_vetted, session_id='RZ034_2024-07-13_str', unit_id=112)

### make spike time array

In [92]:
def generate_spike_counts(spikes, spike_bin_size):
    """
    Generate spike counts and bin centers from spike times.
    returns non-zero spike counts and their corresponding bin centers.
    """
    spike_times = spikes.spike_time.values
    bounds = (
        np.round(spike_times.min(), decimals=1),
        np.round(spike_times.max(), decimals=1)
    )
    spike_bin_edges = np.arange(
        bounds[0] - spike_bin_size,
        bounds[1] + 2*spike_bin_size,
        spike_bin_size
    )
    spike_bin_centers = spike_bin_edges[:-1] + spike_bin_size/2
    spike_counts = np.histogram(spike_times, bins=spike_bin_edges)[0]

    return spike_counts, spike_bin_centers

In [93]:
spike_bin_size = 0.001  # seconds
spike_counts, spike_bin_centers = generate_spike_counts(spikes, spike_bin_size)

In [94]:
spike_counts.shape, spike_bin_centers.shape, spike_counts.sum(), spike_counts.min(), spike_counts.max()

((2443102,), (2443102,), 35683, 0, 2)

### make design matrix

In [95]:
def modify_events(events, trials):
    # Map trial_id to rewarded status
    rewarded_map = trials.set_index('trial_id')['rewarded']

    events_mod = events.copy()
    # Find lick_cons events
    mask_lick_cons = events_mod['event_type'] == 'lick_cons'

    # Map rewarded status for lick_cons events
    rewarded_status = events_mod.loc[mask_lick_cons, 'trial_id'].map(rewarded_map)

    # Update event_type based on rewarded status
    events_mod.loc[mask_lick_cons & (rewarded_status == True), 'event_type'] = 'lick_cons_reward'
    events_mod.loc[mask_lick_cons & (rewarded_status == False), 'event_type'] = 'lick_cons_no_reward'
    return events_mod

In [96]:
event_bin_size = 0.001
duration = int(np.round(spikes.spike_time.values.max(), decimals=1))

events_mod = modify_events(events, trials)

In [97]:
event_rows = events_mod[events_mod.event_type == "lick_wait"]

x_event = np.zeros(int(duration/ event_bin_size))
event_indices = np.round(event_rows.event_start_time.values / event_bin_size).astype(int)
x_event[event_indices] = 1

In [98]:
x_event.shape

(2518000,)

In [105]:
def raised_cosine_basis(n_basis, basis_druation, dt):
    time = np.arange(0, basis_druation, dt)
    centers = np.linspace(0, basis_druation, n_basis)
    width = basis_druation * 1000 / n_basis

    basis_functions = []
    for c in centers:
        phi = np.cos(np.clip((np.pi * (time - c)) / width, -np.pi, np.pi)) * 0.5 + 0.5
        phi[np.abs(time - c) > width] = 0
        basis_functions.append(phi)

    return np.stack(basis_functions), time

In [106]:
# Create basis functions
n_basis = 20
basis_duration = 2  # s
basis_matrix, basis_time = raised_cosine_basis(n_basis, basis_duration, event_bin_size)

In [107]:
basis_matrix.shape

(20, 2000)

In [108]:
# Convolve each basis with the event time series
design_matrix = np.zeros((int(duration/ event_bin_size), n_basis))
for i in range(n_basis):
    convolved = convolve(x_event, basis_matrix[i], mode='full')[:int(duration/ event_bin_size)]
    design_matrix[:, i] = convolved

In [109]:
design_matrix.shape

(2518000, 20)